# 02. 문화 콘텐츠 추천용 마스터 DB 구축

**목적** — 서로 다른 공공 문화 데이터를 하나의 스키마로 통합하고, 추천에 필요한 속성을 태깅

| 항목 | 내용 |
|---|---|
| 입력 | 공공 문화·예술·교육 데이터 21종 (CSV · SHP · Open API) + 직접 큐레이션(온라인 콘텐츠 · DIY 키트) |
| 처리 | 소스별 변환 → 통합 → 주소 표준화 · 좌표 변환 · 가격 파싱 → 중복 제거 → 정합성 보정 → 결측 좌표 보강 |
| 태깅 | `space_type` 활동 공간 · `social_mode` 참여 형태 · `social_intensity_score` 사회적 활동 강도(0~3) · `content_mode` 콘텐츠 특성 |
| 출력 | `recommend_master_db.csv` — 문화 콘텐츠 58,501건, 29개 컬럼, 좌표 보유율 99.95% |

> 원본 데이터는 `../data/raw/`에 두고 실행합니다. API 키는 환경변수로 설정합니다 (`.env.example` 참고).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/데이터분석_포트폴리오/youth-culture-recommendation/notebooks"
!ls ../data | head
!pip install -q -r ../requirements.txt
!apt-get -qq install fonts-nanum > /dev/null

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/데이터분석_포트폴리오/youth-culture-recommendation/notebooks
raw
README.md
recommend_master_db.csv
survey_google.xlsx
survey_modu.xlsx


In [2]:
import os
from google.colab import userdata
for k in ["KCISA_API_KEY", "NAVER_CLIENT_ID", "NAVER_CLIENT_SECRET"]:
    os.environ[k] = userdata.get(k)
print("키 설정 완료")

키 설정 완료


In [3]:
import re
import math
import numpy as np
import pandas as pd

try:
    import geopandas as gpd
except ImportError:
    gpd = None
    print("[경고] geopandas 미설치 — 치유의숲(SHP) 데이터는 건너뜁니다. (pip install geopandas)")

try:
    from pyproj import Transformer
    # 공공데이터포털 인허가 DB(좌표정보(X)(Y))는 보통 EPSG:5174(TM, Bessel 중부원점) 기준입니다.
    # 변환 후 위경도가 한국 범위(위도 33~39, 경도 124~132)를 벗어나면 5181/5179로 바꿔보세요.
    _TM2WGS = Transformer.from_crs("EPSG:5174", "EPSG:4326", always_xy=True)
except ImportError:
    _TM2WGS = None
    print("[경고] pyproj 미설치 — TM좌표(캠핑장 등) 위경도 변환을 건너뜁니다. (pip install pyproj)")

import requests

BASE = "../data/raw"   # 공공데이터 원본 폴더 (저장소 미포함)
MASTER_CSV_PATH = "../data/recommend_master_db.csv"   # 결과물 저장 위치

# API 키는 코드에 직접 쓰지 않고 환경변수로 관리 (.env.example 참고)
KCISA_API_KEY = os.environ.get("KCISA_API_KEY", "")            # 문화공공데이터광장 (예술의전당)
NAVER_CLIENT_ID = os.environ.get("NAVER_CLIENT_ID", "")         # 네이버 클라우드 Geocoding
NAVER_CLIENT_SECRET = os.environ.get("NAVER_CLIENT_SECRET", "")

## 1. 마스터 스키마 · 공통 유틸 (주소 · 가격 · 운영시간 · 좌표 · 태깅)

In [4]:
# 마스터 스키마

MASTER_COLS = [
    "content_id", "source", "large_category", "mid_category", "facility_name",
    "is_program", "program_name",
    "sido", "sigungu", "address",
    "lat", "lon",
    "price_known", "price", "is_free", "price_raw",
    "open_time", "close_time", "is_time_fixed", "operating_hours_remark", "closed_day",
    "tel", "homepage", "description",
    "space_type", "social_mode", "social_intensity_score", "content_mode",
    "last_updated",
]

# 시도명 표준화 -------------------------------------------------
SIDO_NORMALIZE = {
    "강원도": "강원특별자치도", "강원": "강원특별자치도",
    "전북": "전북특별자치도", "전라북도": "전북특별자치도",
    "충북": "충청북도", "충남": "충청남도",
    "전남": "전라남도", "경북": "경상북도", "경남": "경상남도",
    "제주": "제주특별자치도", "제주도": "제주특별자치도",
    "서울": "서울특별시", "부산": "부산광역시", "대구": "대구광역시",
    "인천": "인천광역시", "광주": "광주광역시", "대전": "대전광역시",
    "울산": "울산광역시", "세종": "세종특별자치시", "경기": "경기도",
    # 원본 데이터 자체의 오타 보정 (정합성 검증에서 발견됨)
    "전나남도": "전라남도", "경개도": "경기도", "세종특별시": "세종특별자치시",
}

OFFICIAL_SIDO = {
    "서울특별시", "부산광역시", "대구광역시", "인천광역시", "광주광역시", "대전광역시", "울산광역시",
    "세종특별자치시", "경기도", "강원특별자치도", "충청북도", "충청남도", "전북특별자치도",
    "전라남도", "경상북도", "경상남도", "제주특별자치도",
}
# 시도명 뒤에 공백 없이 시군구가 바로 붙어있는 주소("강원특별자치도양구군")를
# 인식하기 위해, 긴 이름부터 매칭해야 "충청남도"가 "충청북도"보다 먼저 걸리는
# 일이 없도록 길이 내림차순으로 정렬해둡니다.
SIDO_PREFIXES = sorted(OFFICIAL_SIDO, key=len, reverse=True)
SIGUNGU_SUFFIX_RE = re.compile(r"^(.+?(?:시|군|구))")

def norm_sido(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return SIDO_NORMALIZE.get(x, x)

def split_address(addr):
    """주소 문자열에서 시/도, 시/군/구를 추출합니다.
    "강원특별자치도 양구군"처럼 공백이 있는 정상 주소는 토큰 분리로 처리하고,
    "강원특별자치도양구군"처럼 공백 없이 붙은 주소(정합성 검증에서 발견된
    실제 사례)도 표준 시도명을 접두어로 직접 매칭해서 복구합니다."""
    if pd.isna(addr):
        return np.nan, np.nan
    text = str(addr).strip()

    # 표준 시도명이 문자열 맨 앞에 붙어있으면(공백 여부 무관) 그걸로 추출
    for full_name in SIDO_PREFIXES:
        if text.startswith(full_name):
            rest = text[len(full_name):].strip()
            m = SIGUNGU_SUFFIX_RE.match(rest)
            sigungu = m.group(1) if m else (rest.split()[0] if rest.split() else np.nan)
            return full_name, sigungu

    # 표준명으로 못 잡으면 기존처럼 공백 기준 토큰 분리 (구버전 시도 약칭 등 처리)
    tokens = text.split()
    sido = norm_sido(tokens[0]) if len(tokens) > 0 else np.nan
    sigungu = tokens[1] if len(tokens) > 1 else np.nan
    return sido, sigungu

# 수정 ① — program_name 구분자 정리 함수 추가
# "아동미술/성인미술/어르신춤..." 처럼 "/"나 "+"로 그냥 붙어있는 표기를
# description에 이미 적용했던 것과 같은 원칙으로 " · "로 통일합니다.
# 어느 소스(transform_rural, transform_seoul_event, transform_culture_house,
# fetch_sac_programs, transform_arte_program)에서 채워지든 다 똑같이 정리되도록,
# 개별 transform 함수를 다 고치는 대신 master 전체가 합쳐진 뒤 한 번만 적용합니다
# (적용 위치는 맨 아래 build_master_db() 안의 표시 참고).
def clean_program_name(name):
    if not isinstance(name, str):
        return name
    cleaned = re.sub(r"\s*[/+]\s*", " · ", name)
    return cleaned


# 가격 파싱 ------------------------------------------------------
def parse_price(raw):
    """price_known, price(숫자), is_free, price_raw(원문) 반환"""
    if pd.isna(raw):
        return False, np.nan, np.nan, raw
    text = str(raw).strip()
    if text in ("0", "0.0"):
        return True, 0.0, True, raw

    # "장애인무료"처럼 여러 요금 중 일부만 무료인 경우, "무료"라는 글자만
    # 보고 전체를 0원으로 잘못 처리하는 버그가 있었습니다(실제로 "일반1000원+
    # ...+장애인무료" 같은 텍스트가 0원으로 찍힘). 텍스트 안에 실제 금액
    # 표기("N원"/"N만원")가 전혀 없을 때만 "무료"를 확정합니다.
    has_amount = bool(re.search(r"\d[\d,]*\s*(?:만\s*)?원", text))
    if "무료" in text and "유료" not in text and not has_amount:
        return True, 0.0, True, raw

    # "7만원"을 "7원"으로 잘못 읽는 버그가 있었습니다("만" 단위를 고려 안 해서,
    # "7" 바로 뒤가 "원"이 아니라 "만"이라 못 잡고 옛 폴백이 "7"만 집어버림).
    # "N만원" 패턴을 먼저 찾아 10,000을 곱합니다.
    man_won = re.findall(r"\d[\d,]*(?=\s*만\s*원)", text)
    if man_won:
        try:
            val = float(man_won[0].replace(",", "")) * 10000
            return True, val, (val == 0), raw
        except ValueError:
            pass

    # "원"이 바로 뒤에 붙은 숫자("만원" 단위가 아닌 일반 표기, 예: "30,000원")
    won_nums = re.findall(r"\d[\d,]*(?=\s*원)", text)
    if won_nums:
        try:
            val = float(won_nums[0].replace(",", ""))
            return True, val, (val == 0), raw
        except ValueError:
            pass

    # "원"이 전혀 없으면 가격인지 인원수·시간인지("4인실"의 "4" 등) 확신할 수
    # 없어서, 억지로 추측하지 않고 "확인 필요"로 처리합니다 (이전엔 맨 앞
    # 숫자를 무조건 집어서 인원수를 가격으로 잘못 읽는 문제가 있었습니다).
    return False, np.nan, np.nan, raw

# 운영시간 확정여부 ------------------------------------------------
VAGUE_KEYWORDS = ["상이", "별도", "문의", "마을별", "시설별", "협의", "변동", "휴관"]

def parse_hours(open_t, close_t, note=None):
    """open, close, confirmed(bool), note_out(원문 비고) 4개를 반환합니다.
    confirmed=False인 경우, note_out을 UI에 그대로 노출해 '왜 불확실한지'를
    보여줄 수 있습니다 (hours_note 컬럼)."""
    note_out = note if pd.notna(note) else np.nan
    if pd.isna(open_t) or pd.isna(close_t):
        return np.nan, np.nan, False, note_out
    open_t, close_t = str(open_t), str(close_t)

    # "00:00~00:00" + 비고에 "휴관"류 표현 — 원본 데이터가 "운영시간 정보
    # 없음/휴관"을 빈칸이 아니라 00:00이라는 가짜 시각으로 표시해놓은 패턴입니다
    # (도서관 데이터에서 다수 발견됨). 실제로 자정에 열고 자정에 닫는 시설은
    # 거의 없으니, 이 경우는 진짜 시각이 아니라 "정보 없음"으로 보고 결측 처리합니다.
    if open_t == "00:00" and close_t == "00:00" and isinstance(note_out, str) and \
            any(k in note_out for k in VAGUE_KEYWORDS):
        return np.nan, np.nan, False, note_out

    if isinstance(note_out, str) and any(k in note_out for k in VAGUE_KEYWORDS):
        return open_t, close_t, False, note_out
    return open_t, close_t, True, note_out

# TM좌표 -> 위경도 변환 -------------------------------------------
def tm_to_wgs84(x, y):
    if _TM2WGS is None or pd.isna(x) or pd.isna(y):
        return np.nan, np.nan
    try:
        lon, lat = _TM2WGS.transform(float(x), float(y))
        return lat, lon
    except Exception:
        return np.nan, np.nan


# Haversine 거리(km) — 도시공원 vs 국공립공원처럼 같은 시설이 서로 다른
# source에 미세하게 다른 좌표로 등록된 경우를 잡아내는 2차 중복제거용
def haversine_km(lat1, lon1, lat2, lon2):
    if any(pd.isna(v) for v in [lat1, lon1, lat2, lon2]):
        return np.nan
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlmb / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


# 문화성향 매칭용 태그 -------------------------------------------
# (중분류 -> 공간/사회/감각 유형)
# social_intensity_score : 0(1인전용) / 1(1인가능) / 2(소그룹) / 3(그룹)
# -> 추천엔진에서 고립은둔수준별 상한선(은둔<=1, 고립<=2, 일반<=3)과 비교하는 데 사용
# 기본값 근거(방법2): 한국청소년정책연구원 연구보고 21-R07의 단계적 지원 논의,
# 은둔 청년 심층인터뷰 보도(PC방·도서관 등 제한적 외출 실증사례 48%)를 참고해
# 도서관/나혼자문화시설류를 0~1로 설정. 객관지표 보정(방법1)은 compute_social_score 참고.

TAG_MAP = {
    "휴양림":              {"space_type": "실외", "social_mode": "소그룹",   "social_intensity_score": 2, "content_mode": "감각형"},
    "향토문화유적":         {"space_type": "실외", "social_mode": "1인가능", "social_intensity_score": 1, "content_mode": "사고형"},
    "주말농장":            {"space_type": "실외", "social_mode": "소그룹",   "social_intensity_score": 2, "content_mode": "감각형"},
    "박물관":              {"space_type": "실내", "social_mode": "1인가능", "social_intensity_score": 1, "content_mode": "사고형"},
    "미술관":              {"space_type": "실내", "social_mode": "1인가능", "social_intensity_score": 1, "content_mode": "감각형"},
    "도시공원":            {"space_type": "실외", "social_mode": "1인가능", "social_intensity_score": 1, "content_mode": "감각형"},
    "도서관":              {"space_type": "실내", "social_mode": "1인가능", "social_intensity_score": 1, "content_mode": "사고형"},
    "농어촌체험마을":       {"space_type": "실외", "social_mode": "그룹",     "social_intensity_score": 3, "content_mode": "감각형"},
    "관광지":              {"space_type": "실외", "social_mode": "1인가능", "social_intensity_score": 1, "content_mode": "감각형"},
    "실내문화공간(오락등)":  {"space_type": "실내", "social_mode": "그룹",     "social_intensity_score": 3, "content_mode": "감각형"},
    "공연/행사":            {"space_type": "실내", "social_mode": "소그룹",   "social_intensity_score": 2, "content_mode": "감각형"},
    "전시관":              {"space_type": "실내", "social_mode": "1인가능", "social_intensity_score": 1, "content_mode": "사고형"},
    "영화관":              {"space_type": "실내", "social_mode": "소그룹", "social_intensity_score": 2, "content_mode": "감각형"},
    "문화의집":             {"space_type": "실내", "social_mode": "소그룹",   "social_intensity_score": 2, "content_mode": "사고형"},
    "국공립공원":           {"space_type": "실외", "social_mode": "1인가능", "social_intensity_score": 1, "content_mode": "감각형"},
    "캠핑장":              {"space_type": "실외", "social_mode": "그룹",     "social_intensity_score": 3, "content_mode": "감각형"},
    "도보여행지(골목/스토리)": {"space_type": "실외", "social_mode": "1인가능", "social_intensity_score": 1, "content_mode": "사고형"},
    "나혼자문화시설":        {"space_type": "혼합", "social_mode": "1인전용", "social_intensity_score": 0, "content_mode": "혼합"},
    "치유의숲":             {"space_type": "실외", "social_mode": "1인가능", "social_intensity_score": 1, "content_mode": "감각형"},
    "개인추천인프라":        {"space_type": "혼합", "social_mode": "1인가능", "social_intensity_score": 1, "content_mode": "혼합"},
    "온라인콘텐츠":          {"space_type": "온라인", "social_mode": "1인전용", "social_intensity_score": 0, "content_mode": "혼합"},
    "DIY키트":             {"space_type": "온라인", "social_mode": "1인전용", "social_intensity_score": 0, "content_mode": "감각형"},
    "문화예술교육프로그램": {"space_type": "실내", "social_mode": "소그룹",  "social_intensity_score": 2, "content_mode": "감각형"},
}

def get_tags(mid_category):
    return TAG_MAP.get(mid_category, {
        "space_type": np.nan, "social_mode": np.nan,
        "social_intensity_score": np.nan, "content_mode": np.nan,
    })


# 문화성향 매칭용 - social_intensity_score 보정 (방법1: 객관지표 가산) ---------------------
CAPACITY_THRESHOLD = 300  # 수용인원수 >= 300이면 "사람과 마주칠 확률이 높은 곳"으로 간주

def compute_social_score(base_score, capacity=None):
    """카테고리 기본값(base_score, 방법2: 문헌·실증사례 근거)에서 시작해,
    수용인원수 같은 객관지표가 있는 경우에만 +1을 가산합니다(방법1).
    해당 필드가 없는 소스는 base_score를 그대로 사용합니다 — 데이터가
    없는데 억지로 보정하지 않는다는 원칙입니다."""
    if pd.isna(base_score):
        return np.nan
    score = base_score
    if capacity is not None and pd.notna(capacity):
        try:
            if float(capacity) >= CAPACITY_THRESHOLD:
                score += 1
        except (TypeError, ValueError):
            pass
    return min(3, score)


# 공통 row 빌더 --------------------------------------------------
def build_row(source, large_cat, mid_cat, name, is_program, program_name,
              sido, sigungu, address, lat, lon,
              price_raw=None, hours_open=None, hours_close=None, hours_note=None,
              closed_day=None, phone=None, homepage=None, description=None,
              last_updated=None, capacity=None):
    price_known, price, is_free, price_raw_out = parse_price(price_raw)
    h_open, h_close, h_conf, h_note = parse_hours(hours_open, hours_close, hours_note)
    tags = get_tags(mid_cat)
    social_score = compute_social_score(tags["social_intensity_score"], capacity=capacity)
    return {
        "source": source, "large_category": large_cat, "mid_category": mid_cat,
        "facility_name": name, "is_program": bool(pd.notna(is_program) and is_program),
        "program_name": program_name,
        "sido": norm_sido(sido) if pd.notna(sido) else np.nan,
        "sigungu": sigungu, "address": address,
        "lat": lat, "lon": lon,
        "price_known": price_known, "price": price, "is_free": is_free, "price_raw": price_raw_out,
        "open_time": h_open, "close_time": h_close, "is_time_fixed": h_conf, "operating_hours_remark": h_note,
        "closed_day": closed_day,
        "tel": phone, "homepage": homepage, "description": description,
        "space_type": tags["space_type"], "social_mode": tags["social_mode"],
        "social_intensity_score": social_score, "content_mode": tags["content_mode"],
        "last_updated": last_updated,
    }

## 2. 소스별 변환 함수

In [5]:
# 데이터셋별 transform 함수 (19개)

# df1 전국휴양림 (수용인원수 -> capacity 로 social_score 보정에 사용)
def transform_forest(df):
    rows = []
    for _, r in df.iterrows():
        rows.append(build_row(
            source="휴양림", large_cat="자연", mid_cat="휴양림",
            name=r.get("휴양림명"), is_program=False, program_name=np.nan,
            sido=r.get("시도명"), sigungu=np.nan, address=r.get("소재지도로명주소"),
            lat=r.get("위도"), lon=r.get("경도"),
            price_raw=r.get("입장료"),
            phone=r.get("휴양림전화번호"), homepage=r.get("홈페이지주소"),
            description=r.get("주요시설명"), last_updated=r.get("데이터기준일자"),
            capacity=r.get("수용인원수"),
        ))
    return pd.DataFrame(rows)


# df2 전국향토문화유적
def transform_heritage(df):
    rows = []
    for _, r in df.iterrows():
        addr = r.get("소재지도로명주소")
        if pd.isna(addr):
            addr = r.get("소재지지번주소")
        rows.append(build_row(
            source="향토문화유적", large_cat="문화유적", mid_cat="향토문화유적",
            name=r.get("향토문화유적명"), is_program=False, program_name=np.nan,
            sido=np.nan, sigungu=np.nan, address=addr,
            lat=r.get("위도"), lon=r.get("경도"),
            phone=r.get("관리기관전화번호"),
            description=r.get("향토문화유적소개"), last_updated=r.get("데이터기준일자"),
        ))
    return pd.DataFrame(rows)

# df3 전국주말농장
def transform_farm(df):
    rows = []
    for _, r in df.iterrows():
        addr = r.get("소재지도로명주소")
        if pd.isna(addr):
            addr = r.get("소재지지번주소")
        rows.append(build_row(
            source="주말농장", large_cat="체험", mid_cat="주말농장",
            name=r.get("농장명"), is_program=False, program_name=np.nan,
            sido=r.get("시도명"), sigungu=r.get("시군구명"), address=addr,
            lat=r.get("위도"), lon=r.get("경도"),
            price_raw=r.get("분양금액"),
            phone=r.get("전화번호"), last_updated=r.get("데이터기준일자"),
        ))
    return pd.DataFrame(rows)

# df5 전국박물관미술관정보
def transform_museum_art(df):
    rows = []
    for _, r in df.iterrows():
        addr = r.get("소재지도로명주소")
        if pd.isna(addr):
            addr = r.get("소재지지번주소")
        name = str(r.get("시설명", ""))
        mid = "미술관" if "미술관" in name else "박물관"
        rows.append(build_row(
            source="박물관미술관", large_cat="문화시설", mid_cat=mid,
            name=r.get("시설명"), is_program=False, program_name=np.nan,
            sido=np.nan, sigungu=np.nan, address=addr,
            lat=r.get("위도"), lon=r.get("경도"),
            price_raw=r.get("어른관람료"),
            hours_open=r.get("평일관람시작시각"), hours_close=r.get("평일관람종료시각"),
            hours_note=r.get("휴관정보"), closed_day=r.get("휴관정보"),
            phone=r.get("운영기관전화번호"), homepage=r.get("운영홈페이지"),
            description=r.get("박물관미술관소개"), last_updated=r.get("데이터기준일자"),
        ))
    return pd.DataFrame(rows)


# 수정 ③ — 비도시-소공원 등 미조성 계획부지 필터링 함수 추가
# "비도시-소공원5"처럼 실제로 조성이 안 된 행정상 계획부지로 추정되는 공원을
# 걸러냅니다. 원본 CSV에 "조성현황"/"공원구분" 류 컬럼이 있으면 그걸로 정확히
# 거르고, 없으면 이름 패턴(지역구분-이름-숫자)으로 1차 추정만 합니다.
#
def filter_unbuilt_parks(df, name_col="공원명"):
    before = len(df)
    df = df[~df[name_col].astype(str).str.contains(r"\(미조성\)", na=False)]
    removed = before - len(df)
    if removed:
        print(f"[도시공원] '(미조성)' 표시된 공원 {removed}건 제거")
    return df


# df6 전국도시공원정보
def transform_park(df):
    df = filter_unbuilt_parks(df, name_col="공원명")
    rows = []
    for _, r in df.iterrows():
        addr = r.get("소재지도로명주소")
        if pd.isna(addr):
            addr = r.get("소재지지번주소")
        rows.append(build_row(
            source="도시공원", large_cat="자연", mid_cat="도시공원",
            name=r.get("공원명"), is_program=False, program_name=np.nan,
            sido=np.nan, sigungu=np.nan, address=addr,
            lat=r.get("위도"), lon=r.get("경도"),
            phone=r.get("전화번호"), last_updated=r.get("데이터기준일자"),
        ))
    return pd.DataFrame(rows)


# df7 전국도서관표준데이터
def transform_library(df):
    rows = []
    for _, r in df.iterrows():
        rows.append(build_row(
            source="도서관", large_cat="문화시설", mid_cat="도서관",
            name=r.get("도서관명"), is_program=False, program_name=np.nan,
            sido=r.get("시도명"), sigungu=r.get("시군구명"), address=r.get("소재지도로명주소"),
            lat=r.get("위도"), lon=r.get("경도"),
            hours_open=r.get("평일운영시작시각"), hours_close=r.get("평일운영종료시각"),
            hours_note=r.get("휴관일"), closed_day=r.get("휴관일"),
            phone=r.get("도서관전화번호"), homepage=r.get("홈페이지주소"),
            last_updated=r.get("데이터기준일자"),
        ))
    return pd.DataFrame(rows)


# df8 전국농어촌체험휴양마을
def transform_rural(df):
    rows = []
    for _, r in df.iterrows():
        prog = r.get("체험프로그램명")
        rows.append(build_row(
            source="농어촌체험마을", large_cat="체험", mid_cat="농어촌체험마을",
            name=r.get("체험마을명"), is_program=pd.notna(prog), program_name=prog,
            sido=r.get("시도명"), sigungu=r.get("시군구명"), address=r.get("소재지도로명주소"),
            lat=r.get("위도"), lon=r.get("경도"),
            phone=r.get("대표전화번호"), homepage=r.get("홈페이지주소"),
            last_updated=r.get("데이터기준일자"),
        ))
    return pd.DataFrame(rows)


# df9 전국관광지정보 (수용인원수 -> capacity 로 social_score 보정에 사용)
def transform_tourist(df):
    rows = []
    for _, r in df.iterrows():
        addr = r.get("소재지도로명주소")
        if pd.isna(addr):
            addr = r.get("소재지지번주소")
        rows.append(build_row(
            source="관광지", large_cat="자연/관광", mid_cat="관광지",
            name=r.get("관광지명"), is_program=False, program_name=np.nan,
            sido=np.nan, sigungu=np.nan, address=addr,
            lat=r.get("위도"), lon=r.get("경도"),
            phone=r.get("관리기관전화번호"),
            description=r.get("관광지소개"), last_updated=r.get("데이터기준일자"),
            capacity=r.get("수용인원수"),
        ))
    return pd.DataFrame(rows)


# df12 실내문화공간정보
def transform_indoor(df):
    rows = []
    for _, r in df.iterrows():
        rows.append(build_row(
            source="실내문화공간", large_cat="문화시설", mid_cat="실내문화공간(오락등)",
            name=r.get("FCLTY_NM"), is_program=False, program_name=np.nan,
            sido=np.nan, sigungu=np.nan, address=r.get("RDNMADR_NM"),
            lat=r.get("FCLTY_LA"), lon=r.get("FCLTY_LO"),
            homepage=r.get("HMPG_URL"), description=r.get("CTGRY_NM"),
            last_updated=r.get("BASE_DE"),
        ))
    return pd.DataFrame(rows)


# df13 서울시 문화행사 정보
def transform_seoul_event(df):
    # 종료일 기준으로 이미 끝난 과거 행사를 제외하고, 진행중이거나 예정인
    # 것만 남깁니다 (지금까지는 날짜 필터가 전혀 없어서 과거 행사도 다
    # 들어가 있었습니다 — 실제로 빠진 부분이었습니다).
    today = pd.Timestamp.now().normalize()
    end_date = pd.to_datetime(df.get("종료일"), errors="coerce")
    df = df[end_date.isna() | (end_date >= today)]

    rows = []
    for _, r in df.iterrows():
        rows.append(build_row(
            source="서울문화행사", large_cat="공연/행사", mid_cat="공연/행사",
            name=r.get("장소"), is_program=True, program_name=r.get("공연/행사명"),
            sido="서울특별시", sigungu=r.get("자치구"), address=r.get("장소"),
            lat=r.get("위도(X좌표)"), lon=r.get("경도(Y좌표)"),
            price_raw=r.get("이용요금"),
            homepage=r.get("홈페이지?주소"), description=r.get("분류"),
            last_updated=r.get("날짜"),
        ))
    return pd.DataFrame(rows)


# 예술의전당 — 문화공공데이터광장 API(API_TOU_047)에서 직접 가져와, 진행중·예정
# 프로그램만 시설로 등록합니다. 원래 df29(나이대별 예매건수)는 수요통계라서
# 제외했었는데, 같은 API의 title(제목)+period(통계기간) 필드를 이용하면
# "예술의전당에서 언제 무슨 프로그램을 하는지" 정보로도 재활용할 수 있습니다.
# 위치는 예술의전당 단일 시설이라 좌표를 고정값으로 둡니다.
# -------------------------------------------------------------
SAC_ADDRESS = "서울특별시 서초구 남부순환로 2406"
SAC_LAT, SAC_LON = 37.4774, 127.0119

def fetch_sac_programs(max_pages=30):
    if not KCISA_API_KEY:
        raise RuntimeError("KCISA_API_KEY 환경변수가 없어 예술의전당 프로그램을 건너뜁니다")
    url = "https://api.kcisa.kr/openapi/API_TOU_047/request"
    headers = {"accept": "application/json"}
    all_items = []
    page = 1
    while page <= max_pages:
        params = {"serviceKey": KCISA_API_KEY, "numOfRows": 1000, "pageNo": page}
        try:
            resp = requests.get(url, params=params, headers=headers, timeout=10)
            if resp.status_code != 200:
                break
            data = resp.json() or {}
            items = (data.get("response") or {}).get("body", {}).get("items", {}).get("item")
            if not items:
                break
            if isinstance(items, dict):
                items = [items]
            all_items.extend(items)
            page += 1
        except Exception as e:
            print(f"[경고] 예술의전당 API 호출 중단 (page {page}): {e}")
            break

    if not all_items:
        return pd.DataFrame(columns=[c for c in MASTER_COLS if c != "content_id"])

    raw = pd.DataFrame(all_items).drop_duplicates(subset=["title", "period"])

    # period 형식 "YYYYMMDD~YYYYMMDD"에서 종료일을 뽑아, 오늘 이후(진행중/예정)만 남깁니다.
    today = pd.Timestamp.now().normalize()

    def parse_end(period):
        try:
            end_str = str(period).split("~")[-1].strip()
            return pd.to_datetime(end_str, format="%Y%m%d", errors="coerce")
        except Exception:
            return pd.NaT

    raw["_end_date"] = raw["period"].apply(parse_end)
    upcoming = raw[raw["_end_date"].notna() & (raw["_end_date"] >= today)]

    rows = []
    for _, r in upcoming.iterrows():
        rows.append(build_row(
            source="예술의전당", large_cat="공연/행사", mid_cat="공연/행사",
            name="예술의전당", is_program=True, program_name=r.get("title"),
            sido="서울특별시", sigungu="서초구", address=SAC_ADDRESS,
            lat=SAC_LAT, lon=SAC_LON,
            description=r.get("title"), last_updated=str(r.get("period")),
        ))
    return pd.DataFrame(rows)


# 수정 ② — 전시관시설(문화빅데이터) 오염 데이터 제거 함수 추가
# 한의원(나비아이) 등 — 나혼자문화시설의 식당(이북만두)과 같은 종류의 오염.
EXHIBIT_EXCLUDE_KEYWORDS = ["한의원", "병원", "의원", "약국", "동물병원", "치과", "한방"]

def filter_exhibit_pollution(df, name_col="POI_NM"):
    pattern = "|".join(EXHIBIT_EXCLUDE_KEYWORDS)
    before = len(df)
    df = df[~df[name_col].astype(str).str.contains(pattern, na=False)]
    removed = before - len(df)
    if removed:
        print(f"[전시관시설] 무관 시설 {removed}건 제거 (한의원 등)")
    return df


# df14 전시관 시설 데이터 (문화빅데이터플랫폼)
# 영화관 분리 — CL_NM에 브랜드명이 정확히 들어 있어 이름 기반 분리가 안전함
CINEMA_KEYWORDS = ["CGV", "메가박스", "롯데시네마"]

def transform_exhibit(df):
    df = filter_exhibit_pollution(df, name_col="POI_NM")  # 한의원 등은 그대로 제거

    cinema_pattern = "|".join(CINEMA_KEYWORDS)
    is_cinema = df["CL_NM"].astype(str).str.contains(cinema_pattern, na=False)
    print(f"[전시관시설] 영화관 {is_cinema.sum()}건 -> '영화관' 카테고리로 분리")

    rows = []
    for _, r in df.iterrows():
        mid_cat = "영화관" if str(r.get("CL_NM", "")) in CINEMA_KEYWORDS or \
                   any(k in str(r.get("CL_NM", "")) for k in CINEMA_KEYWORDS) else "전시관"
        rows.append(build_row(
            source="전시관시설(문화빅데이터)", large_cat="문화시설", mid_cat=mid_cat,
            name=r.get("POI_NM"), is_program=False, program_name=np.nan,
            sido=r.get("CTPRVN_NM"), sigungu=r.get("SIGNGU_NM"), address=r.get("RDNMADR_NM"),
            lat=r.get("LC_LA"), lon=r.get("LC_LO"),
            description=r.get("MLSFC_NM"), last_updated=r.get("BASE_DE"),
        ))
    return pd.DataFrame(rows)


# df17 문화의 집 시설 및 현황통계
def transform_culture_house(df):
    rows = []
    for _, r in df.iterrows():
        prog = r.get("PROGRM_NM")
        rows.append(build_row(
            source="문화의집", large_cat="문화시설", mid_cat="문화의집",
            name=r.get("FCLTY_NM"), is_program=pd.notna(prog), program_name=prog,
            sido=r.get("CTPRVN_NM"), sigungu=r.get("SIGNGU_NM"), address=r.get("RDNMADR_NM"),
            lat=r.get("FCLTY_LA"), lon=r.get("FCLTY_LO"),
            hours_open=r.get("WORKDAY_OPN_BSNS_TIME"), closed_day=r.get("CLOSEDON_DC"),
            phone=r.get("TEL_NO"), homepage=r.get("HMPG_ADDR"),
            last_updated=r.get("BASE_DE"),
        ))
    return pd.DataFrame(rows)


# df18 국립도립군립 및 도시내 공원
def transform_national_park(df):
    rows = []
    for _, r in df.iterrows():
        rows.append(build_row(
            source="국공립공원", large_cat="자연", mid_cat="국공립공원",
            name=r.get("poi_nm"), is_program=False, program_name=np.nan,
            sido=r.get("sido_nm"), sigungu=r.get("sgg_nm"), address=r.get("rd_nm"),
            lat=r.get("y"), lon=r.get("x"),
            description=r.get("mcate_nm"), last_updated=r.get("base_ymd"),
        ))
    return pd.DataFrame(rows)


# df21 전국야영(캠핑)장업 (TM좌표 -> 위경도 변환 필요)
def transform_camping(df):
    rows = []
    active = df[df["영업상태명"] == "영업/정상"]
    for _, r in active.iterrows():
        lat, lon = tm_to_wgs84(r.get("좌표정보(X)"), r.get("좌표정보(Y)"))
        addr = r.get("도로명주소")
        if pd.isna(addr):
            addr = r.get("지번주소")
        rows.append(build_row(
            source="야영장(캠핑)", large_cat="자연", mid_cat="캠핑장",
            name=r.get("사업장명"), is_program=False, program_name=np.nan,
            sido=np.nan, sigungu=np.nan, address=addr,
            lat=lat, lon=lon,
            phone=r.get("전화번호"), last_updated=r.get("최종수정시점"),
        ))
    return pd.DataFrame(rows)


# df22 도보 여행자를 위한 대중교통 인접 지역문화 관광지
def transform_walk_tour(df):
    rows = []
    for _, r in df.iterrows():
        rows.append(build_row(
            source="도보여행지", large_cat="문화유적/스토리", mid_cat="도보여행지(골목/스토리)",
            name=r.get("AREA_CLTUR_TRRSRT_NM"), is_program=False, program_name=np.nan,
            sido=r.get("CTPRVN_NM"), sigungu=r.get("SIGNGU_NM"), address=r.get("ADDR"),
            lat=r.get("TRRSRT_LA"), lon=r.get("TRRSRT_LO"),
            description=r.get("TRRSRT_STRY_SUMRY_CN"), homepage=r.get("TRRSRT_STRY_URL"),
            last_updated=r.get("OPN_DE"),
        ))
    return pd.DataFrame(rows)


# df23 "나혼자 문화시설 목록"의 LCLAS_NM(대분류) 중, 실제로 문화예술 콘텐츠로
# 볼 수 있는 것만 화이트리스트로 남깁니다. "무장애장소"는 장애인 접근성 정보라
# 식당·편의점까지 섞여있고("이북만두" 사례), "쇼핑시설"은 전통시장이라
# 문화예술복지 처방의 범위를 벗어난다고 판단해 제외했습니다.
SOLO_KEEP_CATEGORIES = ["자연"]

# "자연" 카테고리 안에도 원본 데이터 자체가 동네 공원/어린이공원/소공원 목록을
# 대량으로 포함하고 있어, 이미 도시공원 소스가 다루는 콘텐츠와 중복되고
# "1인전용 특화공간"이라는 카테고리 의미가 옅어지는 문제가 있습니다.
# [시도1: 이름 패턴 필터의 실패] 처음엔 FCLTY_NM에 "공원"이 포함되면 제외했는데,
# "상림", "뒷 말", "제92호"처럼 이름만 보면 공원인지 알 수 없는 코드명·약칭이
# 많아서 필터를 통과해버렸습니다 (실제로 필터링 후에도 남은 행을 보니 전부
# MLSFC_NM(=description)이 "자연_공원"이었습니다). 이름이 아니라 정부가 이미
# 분류해둔 MLSFC_NM 코드 자체로 걸러내는 게 훨씬 정확합니다.
# [참고] 이건 저희가 만든 오류가 아니라 원본 정부데이터 자체의 한계이고,
# "1인전용으로 분류 가능한 특화 문화공간 자체가 공공데이터상 부족하다"는
# 정책적 시사점으로도 보고서에 활용할 수 있는 발견입니다.
SOLO_EXCLUDE_MLSFC_PATTERN = re.compile(r"공원")

def transform_solo(df):  # df23 나혼자 문화시설 목록 (집콕힐링형 핵심 소스)
    df = df[df["LCLAS_NM"].isin(SOLO_KEEP_CATEGORIES)]
    df = df[~df["MLSFC_NM"].astype(str).str.contains(SOLO_EXCLUDE_MLSFC_PATTERN, na=False)]
    rows = []
    for _, r in df.iterrows():
        rows.append(build_row(
            source="나혼자문화시설", large_cat=r.get("LCLAS_NM"), mid_cat="나혼자문화시설",
            name=r.get("FCLTY_NM"), is_program=False, program_name=np.nan,
            sido=r.get("CTPRVN_NM"), sigungu=r.get("SIGNGU_NM"), address=r.get("FCLTY_ROAD_NM_ADDR"),
            lat=r.get("FCLTY_LA"), lon=r.get("FCLTY_LO"),
            phone=r.get("TEL_NO"), description=r.get("MLSFC_NM"),
            last_updated=r.get("UPDT_DT"),
        ))
    return pd.DataFrame(rows)


# 온라인콘텐츠 — 나혼자문화시설(df23)이 정합성 검증 과정에서 실질적으로 0건이
# 된 것을 발견하고, 은둔(score≤1)에게 추천할 1인전용(score=0) 콘텐츠가
# 비어버리는 문제를 보완하기 위해 추가했습니다. 실제 장소가 아니라 온라인
# 서비스라 lat/lon이 없고, 거리·가격 하드필터에서 항상 예외로 처리됩니다
# (recommend_engine.py 쪽에서 처리). URL은 전부 직접 확인한 공식 사이트입니다.
ONLINE_CONTENT_ITEMS = [
    ("국립중앙도서관 전자도서관", "https://www.nl.go.kr", "전자책·원문 자료를 온라인으로 무료 열람"),
    ("국가전자도서관", "https://www.dlibrary.go.kr", "전국 도서관 디지털 자료 통합 검색·열람"),
    ("KMDb 한국고전영화 VOD", "https://www.kmdb.or.kr/vod/main", "한국영상자료원이 제공하는 한국 고전영화 무료 감상"),
    ("Google Arts & Culture", "https://artsandculture.google.com", "전세계 미술관·박물관 소장품 온라인 가상투어"),
    ("국립현대미술관 온라인", "https://www.mmca.go.kr/digitals/digitalMain.do", "무료 온라인 전시"),
    ("국립중앙박물관 온라인전시관", "https://www.museum.go.kr", "소장품 디지털 전시 및 큐레이션 콘텐츠"),
    ("사비나미술관 VR", "https://www.savinamuseum.com/kor/exlist.action?exdgb=VR", "버추얼 미술관 VR 전시 관람"),
    ("e뮤지엄", "https://www.emuseum.go.kr", "전국 박물관 소장품 통합 검색"),
]

def build_online_content():
    rows = []
    for name, url, desc in ONLINE_CONTENT_ITEMS:
        rows.append(build_row(
            source="온라인콘텐츠", large_cat="온라인콘텐츠", mid_cat="온라인콘텐츠",
            name=name, is_program=False, program_name=np.nan,
            sido=np.nan, sigungu=np.nan, address=np.nan,
            lat=np.nan, lon=np.nan,
            price_raw="0",
            homepage=url, description=desc,
        ))
    return pd.DataFrame(rows)


# DIY키트(구매형) — 집콕힐링형+은둔에 추천할 콘텐츠가 부족하다는 피드백을
# 반영해 추가했습니다. 위 온라인콘텐츠(전부 무료)와는 성격이 달라서(구매형)
# 카테고리를 분리했습니다. 사이트 하나에 수천원~수만원대 제품이 다 섞여
# 있어서 "정확한 가격"을 매길 수 없는데, "무료를 원하는 사람에게는 안
# 보이고, 예산이 있는 사람에게는 보인다"만 정확히 맞으면 충분하다고 판단해
# 대표 시작가를 넣었습니다. 처음엔 10,000원으로 뒀는데, 실제로 키트랑·
# 오늘키트 등을 검색해보니 3,800~5,800원짜리 키트가 다수 있어서 10,000원은
# 너무 높은 추정치였습니다("1만원 미만" 응답자에게 "1만원과 같다"로 걸리는
# 것도 애매했고요). 실제 최저가대를 반영해 5,000원으로 낮췄습니다.
DIY_KIT_ITEMS = [
    ("우리두 DIY", "https://uridodiy.com", "DIY 전문몰, 홈데코·인테리어"),
    ("101BOX", "https://101box.co.kr", "창작형 DIY 키트 전문"),
    ("키트랑", "https://kitrang.co.kr", "만들기 DIY 전문몰"),
    ("에듀팜몰", "https://edufarmmall.co.kr/category/diy-%ED%82%A4%ED%8A%B8/144/", "펠트·도자기 등 공예 키트"),
    ("오늘키트", "https://todaykit.co.kr", "마크라메·인테리어소품"),
    ("핸즈코리아", "https://www.hands-kor.co.kr", "만들기 재료·부자재"),
    ("더라임코리아", "https://thelimekorea.co.kr/category/diy%ED%82%A4%ED%8A%B8/92", "DIY 키트 전문"),
    ("가죽공예(myleathertool)", "https://myleathertool.com/category/diy-kit/12/", "가죽공예 전문"),
]
DIY_KIT_REPRESENTATIVE_PRICE = "5000원 (제품마다 다름, 사이트에서 실제 가격 확인 필요)"

def build_diy_kit_content():
    rows = []
    for name, url, desc in DIY_KIT_ITEMS:
        rows.append(build_row(
            source="DIY키트", large_cat="온라인콘텐츠", mid_cat="DIY키트",
            name=name, is_program=False, program_name=np.nan,
            sido=np.nan, sigungu=np.nan, address=np.nan,
            lat=np.nan, lon=np.nan,
            price_raw=DIY_KIT_REPRESENTATIVE_PRICE,
            homepage=url, description=desc,
        ))
    return pd.DataFrame(rows)


# df26 개인추천용 인프라 상세DB (이미 정제된 통합 시설목록)
def transform_personal_infra(df):
    rows = []

    for _, r in df.iterrows():
        rows.append(build_row(
            source="개인추천용인프라DB", large_cat="문화시설", mid_cat="개인추천인프라",
            name=r.get("시설명"), is_program=False, program_name=np.nan,
            sido=r.get("시도명"), sigungu=r.get("자치구"), address=np.nan,
            lat=r.get("위도"), lon=r.get("경도"),
            price_raw=r.get("유무료"), description=r.get("구분"),
        ))

    return pd.DataFrame(rows)


# df28 산림청 치유의 숲 (GeoDataFrame, geometry 중심점 사용)
def transform_healing_forest(gdf):
    if gpd is None or gdf is None:
        return pd.DataFrame(columns=[c for c in MASTER_COLS if c != "content_id"])
    rows = []
    g = gdf.copy()
    try:
        if g.crs is not None and g.crs.to_epsg() != 4326:
            g = g.to_crs(4326)
    except Exception:
        pass
    for _, r in g.iterrows():
        try:
            c = r.geometry.centroid
            lat, lon = c.y, c.x
        except Exception:
            lat, lon = np.nan, np.nan
        rows.append(build_row(
            source="치유의숲", large_cat="자연/치유", mid_cat="치유의숲",
            name=r.get("T_산림청_1"), is_program=False, program_name=np.nan,
            sido=np.nan, sigungu=np.nan, address=r.get("T_산림청_2"),
            lat=lat, lon=lon,
            phone=r.get("T_산림청_3"), homepage=r.get("T_산림청_4"),
            description=r.get("T_산림청_6"),
        ))
    return pd.DataFrame(rows)


# df31, 32 한국문화예술교육진흥원
def transform_arte_program(df32, df31=None):
    """
    df32: 문화예술교육 프로그램 목록 (메인 — 실제 '활동'이 있는 데이터)
    df31: 문화예술교육 단체 목록 (보조 — 공식채널 URL만 매칭해서 보강)
    """
    url_map = {}
    if df31 is not None and "단체명" in df31.columns and "공식채널" in df31.columns:
        url_map = (
            df31.dropna(subset=["공식채널"])
            .drop_duplicates(subset=["단체명"])
            .set_index("단체명")["공식채널"]
            .to_dict()
        )

    rows = []
    for _, r in df32.iterrows():
        facility = r.get("운영장소명")
        if pd.isna(facility):
            facility = r.get("단체이름")

        desc_parts = []
        if pd.notna(r.get("지원사업명")):
            desc_parts.append(str(r["지원사업명"]))
        if pd.notna(r.get("프로그램분야")):
            desc_parts.append(f"분야: {r['프로그램분야']}")
        if pd.notna(r.get("프로그램대상")):
            desc_parts.append(f"대상: {r['프로그램대상']}")
        description = " · ".join(desc_parts) if desc_parts else np.nan

        hours_note = np.nan
        if pd.notna(r.get("운영기간시작")) and pd.notna(r.get("운영기간종료")):
            hours_note = f"운영기간 {r['운영기간시작']}~{r['운영기간종료']}"

        rows.append(build_row(
            source="문화예술교육프로그램", large_cat="문화예술교육", mid_cat="문화예술교육프로그램",
            name=facility, is_program=True, program_name=r.get("프로그램명"),
            sido=np.nan, sigungu=np.nan, address=r.get("운영주소"),
            lat=np.nan, lon=np.nan,  # 주소기반 -> 이후 geocode_missing_coordinates()에서 채움
            price_raw=None,  # 원본에 가격 컬럼이 없어 '확인필요'(price_known=False)로 둠
            hours_open=None, hours_close=None, hours_note=hours_note,
            phone=None, homepage=url_map.get(r.get("단체이름")),
            description=description,
        ))
    return pd.DataFrame(rows)

In [6]:
# 산림청 치유의 숲 좌표 (SHP 공간 데이터) 통합 로드 → df28
def load_healing_forest(base_path=os.path.join(BASE, "산림청_전국 치유의 숲 좌표")):
    if gpd is None:
        return None
    frames = []
    for root_dir, _, files in os.walk(base_path):
        for file in files:
            if file.lower().endswith(".shp"):
                shp_path = os.path.join(root_dir, file)
                try:
                    gdf = gpd.read_file(shp_path)
                    gdf["행정구역코드"] = os.path.basename(root_dir)
                    gdf["원본파일명"] = file
                    frames.append(gdf)
                except Exception as e:
                    print(f"[FAIL] {file}: {e}")
    if not frames:
        print("[건너뜀] 치유의 숲 SHP 파일을 찾지 못했습니다.")
        return None
    out = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True))
    print(f"[OK]   치유의 숲 공간 객체: {len(out)}개")
    return out


df28 = load_healing_forest()
if df28 is None:
    del df28   # build_master_db()가 치유의숲 단계를 건너뛰도록

[OK]   치유의 숲 공간 객체: 38개


In [7]:
# ID 부여 (PLACE_FOREST_0001, PROG_SEOULEVT_0001 형태)

SOURCE_CODE = {
    "휴양림": "FOREST", "향토문화유적": "HERITAGE", "주말농장": "FARM",
    "박물관미술관": "MUSEUM", "도시공원": "PARK", "도서관": "LIB",
    "농어촌체험마을": "RURAL", "관광지": "TOUR", "실내문화공간": "INDOOR",
    "서울문화행사": "SEOULEVT", "전시관시설(문화빅데이터)": "EXPO",
    "문화의집": "CLTHOUSE", "국공립공원": "NPARK", "야영장(캠핑)": "CAMP",
    "도보여행지": "WALK", "나혼자문화시설": "SOLO",
    "개인추천용인프라DB": "INFRA", "치유의숲": "HEALFOREST", "예술의전당": "SAC",
    "온라인콘텐츠": "ONLINE",
    "DIY키트": "DIYKIT",
    "문화예술교육프로그램": "ARTEDU"
}

def assign_ids(df):
    df = df.reset_index(drop=True)
    counters = {}
    ids = []
    for _, r in df.iterrows():
        code = SOURCE_CODE.get(r["source"], "ETC")
        type_prefix = "PROG" if r["is_program"] else "PLACE"
        key = (type_prefix, code)
        counters[key] = counters.get(key, 0) + 1
        ids.append(f"{type_prefix}_{code}_{counters[key]:04d}")
    df["content_id"] = ids
    return df

## 3. 통합 파이프라인

In [8]:
# 메인 파이프라인

def safe_load(path, encoding, transform_fn, label):
    try:
        df = pd.read_csv(path, encoding=encoding) if encoding else pd.read_csv(path)
        out = transform_fn(df)
        print(f"[OK]   {label}: {len(out)} rows")
        return out
    except Exception as e:
        print(f"[FAIL] {label}: {e}")
        return pd.DataFrame(columns=[c for c in MASTER_COLS if c != "content_id"])

def build_master_db():
    loaders = [
        (os.path.join(BASE, "전국휴양림표준데이터.csv"), "cp949", transform_forest, "휴양림"),
        (os.path.join(BASE, "전국향토문화유적표준데이터.csv"), "cp949", transform_heritage, "향토문화유적"),
        (os.path.join(BASE, "전국주말농장표준데이터.csv"), "cp949", transform_farm, "주말농장"),
        (os.path.join(BASE, "전국박물관미술관정보표준데이터.csv"), "cp949", transform_museum_art, "박물관미술관"),
        (os.path.join(BASE, "전국도시공원정보표준데이터.csv"), "cp949", transform_park, "도시공원"),
        (os.path.join(BASE, "전국도서관표준데이터.csv"), "cp949", transform_library, "도서관"),
        (os.path.join(BASE, "전국농어촌체험휴양마을표준데이터.csv"), "cp949", transform_rural, "농어촌체험마을"),
        (os.path.join(BASE, "전국관광지정보표준데이터.csv"), "cp949", transform_tourist, "관광지"),
        (os.path.join(BASE, "실내문화공간정보.csv"), None, transform_indoor, "실내문화공간"),
        (os.path.join(BASE, "서울시 문화행사 정보.csv"), "cp949", transform_seoul_event, "서울문화행사"),
        (os.path.join(BASE, "문화빅데이터플랫폼_국내 문화체육관광 분야 전시관 시설 데이터.csv"), None, transform_exhibit, "전시관시설"),
        (os.path.join(BASE, "문화빅데이터플랫폼_국내 문화체육관광 분야 문화의 집 시설 및 현황통계 데이터.csv"), None, transform_culture_house, "문화의집"),
        (os.path.join(BASE, "문화빅데이터플랫폼_국내 문화체육관광 분야 국립도립군립 및 도시내 공원 데이터.csv"), None, transform_national_park, "국공립공원"),
        (os.path.join(BASE, "문화_일반야영장업.csv"), "cp949", transform_camping, "캠핑장"),
        (os.path.join(BASE, "도보 여행자를 위한 대중교통 인접 지역문화 관광지.csv"), None, transform_walk_tour, "도보여행지"),
        (os.path.join(BASE, "나혼자 문화시설 목록.csv"), None, transform_solo, "나혼자문화시설"),
        # 개인추천용인프라DB는 여기서 제외하고 맨 마지막에 따로 처리합니다 (아래 설명 참고)
    ]

    frames = []
    for path, enc, fn, label in loaders:
        frames.append(safe_load(path, enc, fn, label))

    # 치유의숲(SHP): 앞 셀에서 만든 df28이 있으면 포함, 없으면 건너뜀
    try:
        healing_forest = transform_healing_forest(df28)
        frames.append(healing_forest)
        print(f"[OK]   치유의숲: {len(healing_forest)} rows")
    except NameError:
        print("[건너뜀] 치유의숲: df28이 없어 건너뜁니다 (SHP 로드 셀 확인)")

    # 예술의전당: 문화공공데이터광장 API (키가 없거나 네트워크 오류가 나도 파이프라인은 계속 진행)
    try:
        sac = fetch_sac_programs()
        frames.append(sac)
        print(f"[OK]   예술의전당: {len(sac)} rows (진행중·예정 프로그램만)")
    except Exception as e:
        print(f"[건너뜀] 예술의전당: {e}")


    # 문화예술교육프로그램
    try:
        df31 = pd.read_csv(
            os.path.join(BASE, "한국문화예술교육진흥원_문화예술교육 단체 목록_20251201.csv"),
            encoding="cp949",
        )
        df32 = pd.read_csv(
            os.path.join(BASE, "한국문화예술교육진흥원_문화예술교육 프로그램 목록_20251201.csv"),
            encoding="cp949",
        )
        arte = transform_arte_program(df32, df31)
        frames.append(arte)
        print(f"[OK]   문화예술교육프로그램: {len(arte)} rows")
    except Exception as e:
        print(f"[FAIL] 문화예술교육프로그램: {e}")



    # 온라인콘텐츠: 원본 CSV가 없는 직접 큐레이션 소스라 try/except 없이 바로 추가합니다.
    online = build_online_content()
    frames.append(online)
    print(f"[OK]   온라인콘텐츠: {len(online)} rows")

    diy = build_diy_kit_content()
    frames.append(diy)
    print(f"[OK]   DIY키트: {len(diy)} rows")


    # 개인추천용인프라DB는 여러 카테고리가 뭉뚱그려진 "이미 정제된 통합 시설목록"이라
    # 가장 마지막에 합칩니다. 위 17개 소스 중 어딘가에 이미 같은 시설(이름+좌표)이
    # 있으면 그 구체적인 mid_category(예: 치유의숲, 도서관 등)가 우선 살고,
    # 개인추천용인프라DB는 "다른 어디에도 없는 시설"만 채워주는 역할을 하게 됩니다.
    # (순서를 반대로 하면, 구체적으로 분류된 시설이 뭉뚱그린 카테고리로 덮여버려서
    # 추천엔진의 CLUSTER_POOL 가점을 못 받는 문제가 생깁니다 — 실제로 겪었던 버그입니다.)
    frames.append(safe_load(
        os.path.join(BASE, "개인추천용_인프라_상세DB.csv"), None, transform_personal_infra, "개인추천용인프라"
    ))

    master = pd.concat(frames, ignore_index=True)
    master = master.reindex(columns=[c for c in MASTER_COLS if c != "content_id"])


    # program_name 전체 정리
    # source 상관없이 master 전체가 합쳐진 직후, 한 번에 정리합니다.
    master["program_name"] = master["program_name"].apply(clean_program_name)


    # 주소 기반 시도/시군구 보강
    mask = master["sigungu"].isna() & master["address"].notna()
    extracted = master.loc[mask, "address"].apply(split_address)
    master.loc[mask, "sido"] = master.loc[mask, "sido"].fillna(extracted.apply(lambda t: t[0]))
    master.loc[mask, "sigungu"] = extracted.apply(lambda t: t[1])

    master = assign_ids(master)
    master = master[MASTER_COLS]

    # 이름+위경도(소수 3자리)+프로그램명 기준 중복 제거
    # [중요] program_name을 키에 포함하지 않으면, 같은 시설(예: 예술의전당)에서
    # 열리는 서로 다른 여러 프로그램이 "이름+좌표가 같다"는 이유로 전부 같은
    # 중복으로 처리돼버립니다 (실제로 예술의전당 50건이 1건으로, 서울문화행사
    # 382건이 303건으로 줄어드는 걸 발견하고 찾은 버그입니다). program_name이
    # 없는 시설(place)끼리는 빈 문자열로 동일하게 처리되어 기존 동작 그대로
    # 유지되고, 프로그램끼리는 이름이 달라야 서로 다른 것으로 구분됩니다.
    # pandas 버전과 무관하게 NaN 좌표(온라인콘텐츠 등)가 키 전체를 NaN으로 만들지 않도록 문자열로 고정
    fmt3 = lambda s: s.map(lambda v: "" if pd.isna(v) else f"{float(v):.3f}")
    key = (
        master["facility_name"].fillna("").astype(str).str.strip() + "_" +
        fmt3(master["lat"]) + "_" + fmt3(master["lon"]) + "_" +
        master["program_name"].fillna("").astype(str).str.strip()
    )
    before = len(master)
    master = master.loc[~key.duplicated()]
    print(f"\n중복 제거: {before} -> {len(master)} rows")

    # 2차 중복제거: 도시공원 vs 국공립공원처럼 서로 다른 source가 같은 공원을
    # 미세하게 다른 좌표로 등록한 경우 (1차 중복제거의 소수3자리 반올림 기준을
    # 피해가는 케이스). 더 상세한 정보를 갖고 있는 도시공원(행안부 표준데이터)
    # 쪽을 남기고 국공립공원 쪽을 제거합니다.
    # [임계값 0.2km로 설정한 이유] 처음 0.1km(100m)로 시도했더니 "옥하공원"
    # (전남 고흥)처럼 100m를 살짝 넘는 진짜 중복이 남는 사례가 발견됐습니다.
    # 100~250m 구간을 따로 뽑아보니 233건이 100m 바로 위(100~104m)에 빽빽하게
    # 몰려있었는데, 이름이 우연히 겹치는 별개의 공원이었다면 전국에 흩어져
    # 있어야 정상이라(실제로 전체 동명 공원의 평균 거리는 165km대) — 이 233건은
    # 측정 오차로 좌표가 살짝 어긋난 "진짜 같은 공원"일 가능성이 높다고
    # 판단해 0.2km로 늘렸습니다. (트레이드오프: 같은 동네에 동명 공원이 실제로
    # 2개 있는 드문 경우는 잘못 합쳐질 수 있습니다 — 추천 품질에 큰 영향은 없는
    # 선택이라고 보고 진행합니다.)
    PARK_DEDUP_THRESHOLD_KM = 0.2
    park_a = master[master["source"] == "도시공원"][["facility_name", "lat", "lon"]]
    park_b = master[master["source"] == "국공립공원"][["content_id", "facility_name", "lat", "lon"]]
    if not park_a.empty and not park_b.empty:
        merged = park_b.merge(park_a, on="facility_name", suffixes=("_b", "_a"))
        merged["_dist"] = merged.apply(
            lambda r: haversine_km(r["lat_b"], r["lon_b"], r["lat_a"], r["lon_a"]), axis=1
        )
        drop_ids = merged.loc[merged["_dist"] <= PARK_DEDUP_THRESHOLD_KM, "content_id"].unique().tolist()
        before2 = len(master)
        master = master[~master["content_id"].isin(drop_ids)]
        print(f"도시공원/국공립공원 2차 중복 제거: {before2} -> {len(master)} rows ({len(drop_ids)}건 제거)")

    out_path = MASTER_CSV_PATH

    # =============================================================
    # 최종 정합성 보정 (validate 단계에서 발견된 이슈들)
    # =============================================================
    # 0) 군위군 행정구역 변경(2023.7) 반영: 원본 정부 데이터가 옛 행정구역
    # (경상북도)을 그대로 갖고 있는 소스가 있어서, sigungu가 "군위군"인 모든
    # 행을 sido=대구광역시로 강제 보정합니다. 시도 치환(경상북도→대구) 방식은
    # 군위군이 아닌 다른 경상북도 지역까지 다 바꿔버려서 위험하고, 다른
    # 시도/시군구 처리보다 먼저 실행하면 그 뒤의 주소기반 fallback이 다시
    # 덮어써버리는 순서 문제가 있어 — 모든 처리 순서와 무관하게 항상 적용되도록
    # 맨 마지막에 시군구 기준으로 한 번 더 보정합니다.
    master.loc[master["sigungu"] == "군위군", "sido"] = "대구광역시"

    # 1) sido 잔여 오류 정리: SIDO_NORMALIZE/split_address 개선으로도 못 잡는
    #    케이스(예: 주소에 시도 자체가 없는 "의성군")는 시군구처럼 보이면
    #    sigungu로 옮기고 sido는 비웁니다 (잘못된 시도값을 남기는 것보다
    #    비워두는 게 안전하다는 판단).
    bad_sido_mask = master["sido"].notna() & ~master["sido"].isin(OFFICIAL_SIDO)
    looks_like_sigungu = master.loc[bad_sido_mask, "sido"].astype(str).str.match(r".+(시|군|구)$")
    move_idx = master.loc[bad_sido_mask].loc[looks_like_sigungu].index
    fill_idx = move_idx[master.loc[move_idx, "sigungu"].isna()]
    master.loc[fill_idx, "sigungu"] = master.loc[fill_idx, "sido"]
    n_bad_sido = bad_sido_mask.sum()
    master.loc[bad_sido_mask, "sido"] = np.nan
    if n_bad_sido:
        print(f"[정합성 보정] 표준 17개 시도명에 없는 sido 값 {n_bad_sido}건 정리 "
              f"(가능하면 시군구로 이동, 아니면 비움)")

    # 2) price 이상치: "세계조개박물관"/"1004섬수석미술관"이 똑같이 1,000,020원으로
    #    찍힌 건 원본 박물관미술관 표준데이터 자체의 오류값으로 보입니다
    #    (서로 다른 두 시설이 입장료가 똑같이 100만원대로 찍힐 수는 없음).
    bad_price_mask = master["price"] == 1000020
    n_bad_price = bad_price_mask.sum()
    if n_bad_price:
        master.loc[bad_price_mask, "price"] = np.nan
        master.loc[bad_price_mask, "price_known"] = False
        master.loc[bad_price_mask, "is_free"] = np.nan
        print(f"[정합성 보정] price=1,000,020원 오류값 {n_bad_price}건 -> 미확인으로 처리")

    # 3) 좌표가 한국 영토 범위(위도 33~39, 경도 124~132) 밖이면 위경도를 비웁니다
    #    (잘못된 좌표를 남겨서 거리계산이 조용히 틀어지는 것보다 결측으로
    #    처리해 위치기반 필터링에서 자연히 제외되게 하는 게 안전합니다).
    out_of_kr = master["lat"].notna() & master["lon"].notna() & (
        ~master["lat"].between(33, 39) | ~master["lon"].between(124, 132)
    )
    n_out = out_of_kr.sum()
    if n_out:
        master.loc[out_of_kr, ["lat", "lon"]] = np.nan
        print(f"[정합성 보정] 한국 영토 범위 밖 좌표 {n_out}건 -> 결측 처리")

    # 4) is_free를 pandas의 nullable boolean 타입으로 정리 (True/False/NA를
    #    object가 아니라 깨끗한 bool 계열 타입으로 — 개발자 전달용 정리)
    master["is_free"] = master["is_free"].astype("boolean")

    # 5) open_time/close_time 표기 통일: "9:00"과 "09:00"처럼 소스마다 시(時)에
    # 0을 채우는 방식이 달라 같은 시각이 다른 문자열로 보이는 문제가 있었습니다.
    # 항상 두 자리로 통일합니다 ("00:00"은 그대로 둡니다 — 실제 24시간 운영인지
    # 휴관/정보없음 표시인지는 source별로 직접 확인이 필요해서 여기서 임의로
    # 바꾸지 않습니다).
    def _pad_time(t):
        if pd.isna(t):
            return t
        m = re.match(r"^(\d{1,2}):(\d{2})$", str(t).strip())
        if m:
            h, mi = m.groups()
            return f"{int(h):02d}:{mi}"
        return t

    master["open_time"] = master["open_time"].apply(_pad_time)
    master["close_time"] = master["close_time"].apply(_pad_time)

    # 6) description의 "+" 구분자를 가운뎃점으로 정리 (휴양림 주요시설명 등
    # 원본이 "시설A+시설B+시설C" 형태라 UI에 그대로 노출하면 부자연스럽습니다).
    master["description"] = master["description"].apply(
        lambda x: x.replace("+", " · ") if isinstance(x, str) else x
    )

    # 7) 시설명에 "(휴관)"/"(폐관)"이 붙은 이미 폐쇄된 시설 제외 (원본 데이터에
    # 운영시간 등 다른 정보가 남아있어도, 실제로 갈 수 없는 곳을 추천하면 안 됨).
    closed_name_mask = master["facility_name"].astype(str).str.contains(r"\(휴관\)|\(폐관\)", na=False)
    n_closed = closed_name_mask.sum()
    if n_closed:
        master = master[~closed_name_mask]
        print(f"[정합성 보정] 시설명에 (휴관)/(폐관) 표시된 폐쇄 시설 {n_closed}건 제외")


    master.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"저장 완료: {out_path}")
    print(f"총 {len(master)} rows / {master['source'].nunique()} 개 소스")
    print(master["source"].value_counts())
    return master

In [9]:
# 네이버 클라우드 Geocoding API로 결측 좌표 보강
# - 주소가 있지만 위경도가 없는 시설만 대상
# - 응답 좌표가 한국 범위(위도 33~39, 경도 124~132) 밖이면 잘못된 매칭으로 보고 제외
# - API 키는 환경변수 NAVER_CLIENT_ID / NAVER_CLIENT_SECRET 사용
import os
import time
import requests
import numpy as np
import pandas as pd

GEOCODE_URL = "https://maps.apigw.ntruss.com/map-geocode/v2/geocode"  # 신형 Maps 서비스 엔드포인트

REQUEST_DELAY_SEC = 0.15  # 호출 간 대기시간 (API 제한 보호용, 필요시 조정)


def clean_address(address_text):
    """쉼표 이후(부가설명)나 "OO 외"(여러 곳 묶음 표기)를 잘라내 지오코딩이
    더 잘 인식하는 단일 주소 형태로 정제합니다."""
    if pd.isna(address_text) or not str(address_text).strip() or str(address_text).lower() == "nan":
        return None
    return str(address_text).split(",")[0].split(" 외")[0].strip()


def geocode_address(address):
    """주소 문자열 -> (lat, lon). 실패하거나 한국 범위 밖이면 (nan, nan)."""
    clean_addr = clean_address(address)
    if not clean_addr:
        return np.nan, np.nan

    headers = {
        "X-NCP-APIGW-API-KEY-ID": NAVER_CLIENT_ID,
        "X-NCP-APIGW-API-KEY": NAVER_CLIENT_SECRET,
    }
    # requests의 params= 를 사용해 한글/특수문자 URL 인코딩을 자동으로 처리합니다
    # (URL에 직접 문자열을 이어붙이면 공백·한글이 깨질 수 있어서 더 안전한 방식입니다).
    params = {"query": clean_addr}
    try:
        resp = requests.get(GEOCODE_URL, headers=headers, params=params, timeout=5)
        if resp.status_code != 200:
            print(f"[실패:{resp.status_code}] {clean_addr}")
            return np.nan, np.nan
        data = resp.json()
        addresses = data.get("addresses", [])
        if not addresses:
            print(f"[결과없음] {clean_addr}")
            return np.nan, np.nan
        top = addresses[0]
        lat, lon = float(top["y"]), float(top["x"])  # 네이버 API: x=경도, y=위도
        if not (33 <= lat <= 39 and 124 <= lon <= 132):
            print(f"[범위밖 의심] {clean_addr} -> lat={lat}, lon={lon}")
            return np.nan, np.nan
        return lat, lon
    except Exception as e:
        print(f"[에러] {clean_addr}: {e}")
        return np.nan, np.nan


def geocode_missing_coordinates():
    if not (NAVER_CLIENT_ID and NAVER_CLIENT_SECRET):
        raise RuntimeError("NAVER_CLIENT_ID / NAVER_CLIENT_SECRET 환경변수를 설정하세요")
    master = pd.read_csv(MASTER_CSV_PATH)

    target_mask = master["lat"].isna() & master["lon"].isna() & master["address"].notna()
    target = master[target_mask]
    print(f"좌표 보강 대상: {len(target)}건 (전체 {len(master)}건 중)")

    filled, failed = 0, 0
    for idx, row in target.iterrows():
        lat, lon = geocode_address(row["address"])
        if pd.notna(lat) and pd.notna(lon):
            master.loc[idx, "lat"] = lat
            master.loc[idx, "lon"] = lon
            filled += 1
        else:
            failed += 1
        time.sleep(REQUEST_DELAY_SEC)

    print(f"\n복구 완료: {filled}건 / 실패(주소 못찾음 등): {failed}건")
    master.to_csv(MASTER_CSV_PATH, index=False, encoding="utf-8-sig")
    print(f"저장 완료: {MASTER_CSV_PATH}")

    still_missing = master["lat"].isna().sum()
    print(f"여전히 좌표 결측인 행: {still_missing}건 "
          f"(주소 자체가 없는 경우라 지오코딩으로 복구 불가능한 케이스입니다)")
    return master

In [10]:
master = build_master_db()               # 마스터 DB 생성 → recommend_master_db.csv
master = geocode_missing_coordinates()   # 결측 좌표 보강 (같은 파일에 덮어쓰기)

[OK]   휴양림: 183 rows
[OK]   향토문화유적: 4271 rows
[OK]   주말농장: 7 rows
[OK]   박물관미술관: 1068 rows
[도시공원] '(미조성)' 표시된 공원 14건 제거
[OK]   도시공원: 18378 rows
[OK]   도서관: 3554 rows
[OK]   농어촌체험마을: 1254 rows
[OK]   관광지: 854 rows
[OK]   실내문화공간: 4239 rows
[OK]   서울문화행사: 48 rows
[전시관시설] 무관 시설 16건 제거 (한의원 등)
[전시관시설] 영화관 466건 -> '영화관' 카테고리로 분리
[OK]   전시관시설: 11345 rows
[OK]   문화의집: 45 rows
[OK]   국공립공원: 9961 rows
[OK]   캠핑장: 3540 rows
[OK]   도보여행지: 12967 rows
[OK]   나혼자문화시설: 0 rows
[OK]   치유의숲: 38 rows
[경고] 예술의전당 API 호출 중단 (page 15): 'NoneType' object has no attribute 'get'
[OK]   예술의전당: 136 rows (진행중·예정 프로그램만)
[OK]   문화예술교육프로그램: 1311 rows
[OK]   온라인콘텐츠: 8 rows
[OK]   DIY키트: 8 rows
[OK]   개인추천용인프라: 12502 rows

중복 제거: 85717 -> 59121 rows
도시공원/국공립공원 2차 중복 제거: 59121 -> 58427 rows (694건 제거)
[정합성 보정] 표준 17개 시도명에 없는 sido 값 221건 정리 (가능하면 시군구로 이동, 아니면 비움)
[정합성 보정] 한국 영토 범위 밖 좌표 3건 -> 결측 처리
[정합성 보정] 시설명에 (휴관)/(폐관) 표시된 폐쇄 시설 53건 제외
저장 완료: ../data/recommend_master_db.csv
총 58374 rows / 21 개 소스
source
도시공원             

/tmp/ipykernel_4084/2149496184.py:61: DtypeWarning: Columns (16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  master = pd.read_csv(MASTER_CSV_PATH)


좌표 보강 대상: 1674건 (전체 58374건 중)
[결과없음] 인천광역시 옹진군 자월면 자원리 1116-2
[결과없음] 강원특별자치도 양양군 서면 용소리 335
[결과없음] 남구 3.1만세운동길 6 이강하미술관
[결과없음] 나주시 나주천1길 나주나빌레라문화센터
[결과없음] 나주시 나주천1길 나주나빌레라문화센터
[결과없음] 경기도 고양시 일산서구 중앙로 지하1431
[결과없음] 경기도 고양시 일산서구 중앙로 지하1431
[결과없음] 완주군 용진읍 462-9
[결과없음] 완주군 용진읍 462-9
[결과없음] 완주군 용진읍 462-9
[결과없음] 완주군 용진읍 462-9

복구 완료: 1663건 / 실패(주소 못찾음 등): 11건
저장 완료: ../data/recommend_master_db.csv
여전히 좌표 결측인 행: 28건 (주소 자체가 없는 경우라 지오코딩으로 복구 불가능한 케이스입니다)


## 4. 정합성 검증

In [11]:
park = master[master["source"] == "도시공원"][["facility_name", "lat", "lon"]].rename(
    columns={"lat": "lat_park", "lon": "lon_park"})
npark = master[master["source"] == "국공립공원"][["facility_name", "lat", "lon"]].rename(
    columns={"lat": "lat_npark", "lon": "lon_npark"})

merged = pd.merge(park, npark, on="facility_name", how="inner")
print(f"이름이 겹치는 공원: {len(merged)}건 (도시공원 {len(park)}개, 국공립공원 {len(npark)}개 중)")

merged["dist_km"] = merged.apply(
    lambda r: haversine_km(r["lat_park"], r["lon_park"], r["lat_npark"], r["lon_npark"]), axis=1)
print(merged["dist_km"].describe())

# 100m 이내 = 같은 공원일 가능성이 높은데 1차 중복 제거를 통과한 경우
close_dup = merged[merged["dist_km"] < 0.1]
print(f"\n100m 이내(동일 시설 의심): {len(close_dup)}건")
print(close_dup.head(20))

이름이 겹치는 공원: 9141건 (도시공원 17705개, 국공립공원 7922개 중)
count    9141.000000
mean      178.366325
std       117.104891
min         0.200416
25%        81.507893
50%       172.200967
75%       265.562670
max       582.173500
Name: dist_km, dtype: float64

100m 이내(동일 시설 의심): 0건
Empty DataFrame
Columns: [facility_name, lat_park, lon_park, lat_npark, lon_npark, dist_km]
Index: []


In [12]:
print("="*60)
print("1. 기본 정보")
print("="*60)
print(f"전체 행 수: {len(master):,} / 컬럼 수: {master.shape[1]}")
print(master.dtypes)

print("\n" + "="*60)
print("2. content_id 중복 체크 (0건이어야 정상)")
print("="*60)
print(f"중복 content_id: {master['content_id'].duplicated().sum()}건")

print("\n" + "="*60)
print("3. 시도(sido) 검증 — 공식 17개 시도명과 비교")
print("="*60)
OFFICIAL_SIDO = {
    "서울특별시","부산광역시","대구광역시","인천광역시","광주광역시","대전광역시","울산광역시",
    "세종특별자치시","경기도","강원특별자치도","충청북도","충청남도","전북특별자치도",
    "전라남도","경상북도","경상남도","제주특별자치도",
}
actual_sido = set(master["sido"].dropna().unique())
print("공식 목록에 없는 이상한 시도명:", actual_sido - OFFICIAL_SIDO)
print(f"\nsido 결측치: {master['sido'].isnull().sum()}건 ({master['sido'].isnull().mean()*100:.1f}%)")
print("\n시도별 row 수:")
print(master["sido"].value_counts(dropna=False))

print("\n" + "="*60)
print("4. 시군구(sigungu) 검증 — 이상하게 짧거나 숫자만 있는 값")
print("="*60)
weird_sigungu = master[
    master["sigungu"].notna() &
    (master["sigungu"].astype(str).str.len() <= 1)
]["sigungu"].unique()
print("이상하게 짧은 시군구 값:", weird_sigungu)
print(f"sigungu 결측치: {master['sigungu'].isnull().sum()}건 ({master['sigungu'].isnull().mean()*100:.1f}%)")

print("\n" + "="*60)
print("5. 위경도(lat/lon) — 한국 영토 범위(위도 33~39, 경도 124~132) 밖 체크")
print("="*60)
out_of_korea = master[
    master["lat"].notna() & master["lon"].notna() &
    (~master["lat"].between(33, 39) | ~master["lon"].between(124, 132))
]
print(f"한국 범위 밖 좌표: {len(out_of_korea)}건")
if len(out_of_korea) > 0:
    print(out_of_korea[["source", "facility_name", "lat", "lon"]].head(10))
print(f"\n위경도 결측치: lat {master['lat'].isnull().sum()}건 / lon {master['lon'].isnull().sum()}건")

print("\n" + "="*60)
print("6. price — 음수나 비정상적으로 큰 값 체크")
print("="*60)
weird_price = master[master["price"].notna() & ((master["price"] < 0) | (master["price"] > 1000000))]
print(f"이상한 가격(음수 또는 100만원 초과): {len(weird_price)}건")
if len(weird_price) > 0:
    print(weird_price[["facility_name", "price", "price_raw"]].head(10))

print("\n" + "="*60)
print("7. bool 컬럼 타입 체크 (is_program, is_free, price_known, is_time_fixed)")
print("="*60)
for col in ["is_program", "is_free", "price_known", "is_time_fixed"]:
    print(f"{col}: dtype={master[col].dtype}, 고유값={master[col].unique()}")

print("\n" + "="*60)
print("8. social_intensity_score — 0~3 범위 밖 값 체크")
print("="*60)
weird_score = master[master["social_intensity_score"].notna() & ~master["social_intensity_score"].between(0, 3)]
print(f"범위 밖 값: {len(weird_score)}건")
print(f"score 결측치: {master['social_intensity_score'].isnull().sum()}건")
print(master["social_intensity_score"].value_counts(dropna=False).sort_index())

print("\n" + "="*60)
print("9. mid_category / large_category 고유값 눈으로 확인")
print("="*60)
print("mid_category:", sorted(master["mid_category"].dropna().unique()))
print("\nlarge_category 고유값 수:", master["large_category"].nunique())
print(master["large_category"].value_counts(dropna=False).head(20))

print("\n" + "="*60)
print("10. facility_name 결측/공백 체크 (추천에 필수)")
print("="*60)
print(f"facility_name 결측: {master['facility_name'].isnull().sum()}건")
print(f"facility_name 공백만 있는 값: {(master['facility_name'].astype(str).str.strip()=='').sum()}건")

1. 기본 정보
전체 행 수: 58,374 / 컬럼 수: 29
content_id                 object
source                     object
large_category             object
mid_category               object
facility_name              object
is_program                   bool
program_name               object
sido                       object
sigungu                    object
address                    object
lat                       float64
lon                       float64
price_known                  bool
price                     float64
is_free                    object
price_raw                  object
open_time                  object
close_time                 object
is_time_fixed                bool
operating_hours_remark     object
closed_day                 object
tel                        object
homepage                   object
description                object
space_type                 object
social_mode                object
social_intensity_score      int64
content_mode               object
last_updated 